In [1]:
%load_ext autoreload
%autoreload 2

In [9]:
import pandas as pd

from src.train_ar import ModelWithMetaInfoAr, split
from src.train_prophet import ModelWithMetaInfoProphet
from src.utils import fix_df

In [6]:
data_filepath = 'data/structure/AKMOLA/@regions/Akmola/load/values.parquet'

In [ ]:
df = pd.read_parquet(data_filepath)

if 'name' in df.columns:
    df = df.drop(columns=['name'])
if 'number' in df.columns:
    df = df.drop(columns=['number'])

df = fix_df(df, interpolate=False)
df_train, df_val = split(df, ratio=0.8)

In [4]:
model = ModelWithMetaInfoAr.load_model('xgb-calendar/AKMOLA/@regions/Akmola/load/xgb_model.json')
model

ModelWithMetaInfo({'W_future': 30,
 'W_past': 72,
 'debug': False,
 'early_stopping_rounds': 100,
 'features_info': [FeatureInfo(name='value',
                               span='past',
                               type_='numerical',
                               agg='full',
                               cycle_h=1),
                   FeatureInfo(name='hour',
                               span='future',
                               type_='categorical',
                               agg='first',
                               cycle_h=None),
                   FeatureInfo(name='dayofweek',
                               span='future',
                               type_='categorical',
                               agg='first',
                               cycle_h=None),
                   FeatureInfo(name='month',
                               span='future',
                               type_='categorical',
                               agg='first',
                     

In [5]:
y, y_pred = model.predict(df_val)

/home/mkotyushev/Documents/oik-new/src/train_ar.py:131: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  df['is_holiday'] = df.index.normalize().isin(holidays_kz).astype(int)


In [8]:
y.shape

(5156, 30)

In [15]:
freq = 'D'
agg_func = 'sum'

df = pd.read_parquet(data_filepath)
df = fix_df(df, interpolate=False)
df = df[['value']].sort_index().dropna()

if freq in ['D', 'M']:
    if agg_func in ['sum', 'elect']:
        df = df.resample(freq).sum()
        if agg_func == 'elect':
            df['value'] *= 1000
    elif agg_func == 'mean':
        df = df.resample(freq).mean()
    else:
        raise ValueError(f'Unknown aggregation function: {agg_func}')

df = df[['value']]  # Keep only 'value' column
df = df \
    .reset_index('dt') \
    .rename(columns={'dt': 'ds', 'value': 'y'})

In [16]:
model = ModelWithMetaInfoProphet.load_model('models/prophet_new/elect/D/AKMOLA/@regions/Akmola/generation/prophet_model.json')
model

ModelWithMetaInfo({'debug': False,
 'model': <prophet.forecaster.Prophet object at 0x746a62f0f390>,
 'model_params': {'daily_seasonality': False,
                  'growth': 'flat',
                  'holidays_prior_scale': 10.0,
                  'seasonality_mode': 'additive',
                  'weekly_seasonality': True,
                  'yearly_seasonality': True},
 'verbose': False})

In [17]:
df_preds = model.predict(df)
df_preds

,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,additive_terms,additive_terms_lower,additive_terms_upper,weekly,weekly_lower,weekly_upper,yearly,yearly_lower,yearly_upper,multiplicative_terms,multiplicative_terms_lower,multiplicative_terms_upper,yhat
0,2021-05-26,1.454491e+07,9.012605e+06,1.390819e+07,1.454491e+07,1.454491e+07,-3.027710e+06,-3.027710e+06,-3.027710e+06,-61866.450215,-61866.450215,-61866.450215,-2.965844e+06,-2.965844e+06,-2.965844e+06,0.0,0.0,0.0,1.151720e+07
1,2021-05-27,1.454491e+07,9.013541e+06,1.398838e+07,1.454491e+07,1.454491e+07,-3.090146e+06,-3.090146e+06,-3.090146e+06,-18686.894258,-18686.894258,-18686.894258,-3.071459e+06,-3.071459e+06,-3.071459e+06,0.0,0.0,0.0,1.145476e+07
2,2021-05-28,1.454491e+07,9.064753e+06,1.376327e+07,1.454491e+07,1.454491e+07,-3.138094e+06,-3.138094e+06,-3.138094e+06,39723.479932,39723.479932,39723.479932,-3.177817e+06,-3.177817e+06,-3.177817e+06,0.0,0.0,0.0,1.140681e+07
3,2021-05-29,1.454491e+07,8.886588e+06,1.374310e+07,1.454491e+07,1.454491e+07,-3.334439e+06,-3.334439e+06,-3.334439e+06,-50730.711359,-50730.711359,-50730.711359,-3.283709e+06,-3.283709e+06,-3.283709e+06,0.0,0.0,0.0,1.121047e+07
4,2021-05-30,1.454491e+07,8.877939e+06,1.380539e+07,1.454491e+07,1.454491e+07,-3.255897e+06,-3.255897e+06,-3.255897e+06,131961.528687,131961.528687,131961.528687,-3.387858e+06,-3.387858e+06,-3.387858e+06,0.0,0.0,0.0,1.128901e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1043,2024-04-03,1.454491e+07,1.480058e+07,1.966431e+07,1.454491e+07,1.454491e+07,2.712203e+06,2.712203e+06,2.712203e+06,-61866.450215,-61866.450215,-61866.450215,2.774069e+06,2.774069e+06,2.774069e+06,0.0,0.0,0.0,1.725711e+07
1044,2024-04-04,1.454491e+07,1.464738e+07,1.980710e+07,1.454491e+07,1.454491e+07,2.639989e+06,2.639989e+06,2.639989e+06,-18686.894258,-18686.894258,-18686.894258,2.658676e+06,2.658676e+06,2.658676e+06,0.0,0.0,0.0,1.718490e+07
1045,2024-04-05,1.454491e+07,1.472939e+07,1.947498e+07,1.454491e+07,1.454491e+07,2.573502e+06,2.573502e+06,2.573502e+06,39723.479932,39723.479932,39723.479932,2.533779e+06,2.533779e+06,2.533779e+06,0.0,0.0,0.0,1.711841e+07
1046,2024-04-06,1.454491e+07,1.443965e+07,1.932071e+07,1.454491e+07,1.454491e+07,2.349188e+06,2.349188e+06,2.349188e+06,-50730.711359,-50730.711359,-50730.711359,2.399919e+06,2.399919e+06,2.399919e+06,0.0,0.0,0.0,1.689410e+07
